# 02 — Análise das Features, Pré-processamento Físico e Cinemática em $\kappa_\lambda$

Este notebook examina detalhadamente as **56 variáveis de entrada da Tabela 29 da AN-25-103** (49 contínuas + 7 categóricas):

1. **Auditoria contra vazamento de metadados (Teste 13.1):** garante que peso, label, amostra ou $\kappa_\lambda$ não estejam infiltrados entre as features.
2. **Pré-processamento físico (`apply_physical_preprocessing`):** tratamento de objetos opcionais ausentes (`bjet2`, `hbb`, `htthbb` quando `has_jet_pair == 0`; `fatjet`, `httfatjet` quando `has_fatjet == 0`; e `bjet1` quando não há jato AK4 reconstruído com $E > 0$) e compressão logarítmica com sinal `sign(x)*log1p(|x|)` nas covariâncias da PuppiMET.
3. **Padronização `FeatureScaler` (exclusivamente no treino) e `CategoricalPreprocessor`:** verificação de caudas extremas (`|z| > 50`) e vocabulários dos embeddings.
4. **Validação física dos 4-vetores do LBN (`check_four_vectors`):** confirmação de $E^2 \ge |\vec{p}|^2 - 1\,\text{GeV}^2$ nos objetos presentes e **zeragem exata** $(0, 0, 0, 0)$ de `bjet2` e `fatjet` ausentes após despadronização na camada `FourVectorsFromFeatures`.
5. **Cinemática física ($m_{\tau\tau}, m_{bb}, m_{HH}$) por $\kappa_\lambda$ e AUC univariada (Teste 13.3):** visualização direta do deslocamento de $m_{HH}$ em $\kappa_\lambda = 5.0$ vs $\kappa_\lambda = 2.45$ (AN-25-103 Seção 5.5.3).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.parquet_dataset import check_feature_leakage_names
from src.data.preprocessing import check_four_vectors, prepare_datasets_from_config
from src.evaluation.plotting import plot_feature_distributions
from src.evaluation.roc import univariate_feature_aucs
from src.evaluation.score_distributions import reconstruct_invariant_masses
from src.utils.config import load_config
from src.utils.paths import OutputPaths
from src.utils.seed import set_all_seeds

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

In [ ]:
USE_SYNTHETIC_IF_MISSING = False
cfg = load_config(PROJECT_ROOT / "configs" / "default.yaml")
set_all_seeds(cfg["seed"])
out_paths = OutputPaths.from_root(cfg["paths"]["output_dir"], create=True)

if not Path(cfg["paths"]["parquet_dir"]).exists() and USE_SYNTHETIC_IF_MISSING:
    from src.data.synthetic import generate_synthetic_parquets
    cfg["paths"]["parquet_dir"] = str(generate_synthetic_parquets(PROJECT_ROOT / "data" / "synthetic", cfg, seed=cfg["seed"]))

# 13.1 Checagem de nomes de features contra vazamento
bad_cols = check_feature_leakage_names(
    cfg["features"]["continuous_features"],
    cfg["features"]["categorical_features"],
    weight_col=cfg["features"]["weight_col"],
    forbidden=cfg["features"]["forbidden_substrings"],
)
assert not bad_cols, f"Colunas suspeitas encontradas: {bad_cols}"
print("Teste 13.1 OK: apenas as 49 contínuas + 7 categóricas da Tabela 29 entram na rede.")

## 1. Pipeline Completo de Dados e Padronização Treino-Only

In [ ]:
data = prepare_datasets_from_config(cfg, verbose=True)

print("\nTabela de média e desvio-padrão estimados apenas no conjunto de TREINO (primeiras 15 features):")
display(data.scaler.to_dataframe().head(15))

## 2. Caudas das Variáveis Padronizadas e Consistência dos 4-Vetores para o LBN

In [ ]:
cont_cols = cfg["features"]["continuous_features"]
x_tr = data.x_cont["train"]

z_stats = pd.DataFrame({
    "mean_train": x_tr.mean(axis=0),
    "std_train": x_tr.std(axis=0),
    "p99_9_abs_z": np.percentile(np.abs(x_tr), 99.9, axis=0),
    "max_abs_z": np.abs(x_tr).max(axis=0),
    "frac_zero_%": 100.0 * (x_tr == 0.0).mean(axis=0),
}, index=cont_cols).sort_values("max_abs_z", ascending=False)

print("Top 12 variáveis contínuas por max|z| no treino:")
display(z_stats.head(12))

print("Verificação física dos 4-vetores entregues à camada LBN:")
fv_df = check_four_vectors(
    x_cont=data.x_cont["train"],
    x_cat=data.x_cat["train"],
    scaler=data.scaler,
    continuous_features=cont_cols,
    lbn_particles=cfg["features"]["lbn_particles"],
    mask_bjet1_without_pair=bool(cfg["features"].get("mask_bjet1_always_on_no_pair", False)),
    strict=True,
    verbose=True,
)
display(fv_df)

## 3. Reconstrução de $m_{\tau\tau}$, $m_{bb}$ e $m_{HH}$: Por que Parametrizar em $\kappa_\lambda$?

Abaixo reconstruímos as massas invariantes diretamente dos 4-vetores regressados (`reg_dnn_moe_htt_*`, `hbb_*`, `htthbb_*`). Observe como o espectro de $m_{HH}$ muda drasticamente entre $\kappa_\lambda = 1.0$ (SM), $\kappa_\lambda = 2.45$ e $\kappa_\lambda = 5.0$ (acumulando eventos próximo ao limiar $2 m_H \approx 250\,\text{GeV}$, região onde o fundo $t\bar{t}$ e DY é mais intenso).

In [ ]:
train_df = data.splits["train"]
masses_tr = reconstruct_invariant_masses(train_df)
df_plot = pd.concat([train_df[["label", "kappa_lambda", "w_phys"]], masses_tr], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (mcol, title, xmax) in zip(
    axes,
    [("m_tautau", r"$m_{\tau\tau}^{\mathrm{reg}}$ [GeV]", 260),
     ("m_bb", r"$m_{bb}$ [GeV]", 280),
     ("m_hh", r"$m_{HH}^{\mathrm{reg}}$ [GeV]", 900)],
):
    vals = df_plot[mcol].to_numpy(np.float64)
    w = np.clip(df_plot["w_phys"].to_numpy(np.float64), 0.0, None)
    y = df_plot["label"].to_numpy()
    kl = df_plot["kappa_lambda"].to_numpy(np.float64)
    bins = np.linspace(40 if mcol != "m_hh" else 200, xmax, 50)

    for c, cname in ((0, "DY"), (1, "ttbar")):
        m = (y == c) & np.isfinite(vals)
        ax.hist(vals[m], bins=bins, weights=w[m], density=True, histtype="stepfilled", alpha=0.22, label=cname)
    for k in cfg["physics"]["kappa_grid"]:
        mk = (y == 2) & np.isclose(kl, k) & np.isfinite(vals)
        ax.hist(vals[mk], bins=bins, weights=w[mk], density=True, histtype="step", lw=1.9, label=rf"HH ($\kappa_\lambda={k:g}$)")

    ax.set_xlabel(title)
    ax.set_ylabel("Densidade normalizada")
    ax.set_title(f"Distribuição de {title}")
    ax.grid(alpha=0.3)
    if mcol == "m_tautau":
        ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(out_paths.plots / "kinematics_masses_by_kappa.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Distribuições das Principais Features Cinemáticas e de *b-tagging*

In [ ]:
key_features = [
    "reg_dnn_moe_bjet1_hhbtag",
    "reg_dnn_moe_bjet2_hhbtag",
    "reg_dnn_moe_bjet1_tag_b",
    "reg_dnn_moe_vis_tau1_e",
    "reg_dnn_moe_htt_e",
    "reg_dnn_moe_htthbb_e",
    "reg_dnn_moe_met_px",
    "reg_dnn_moe_met_cov00",
    "reg_dnn_moe_bjet1_e",
]

plot_feature_distributions(
    df=train_df,
    features=key_features,
    class_labels=cfg["physics"]["class_labels"],
    ncols=3,
    save_path=out_paths.plots / "feature_distributions_overview.png",
    show=True,
)

## 5. Distribuição das 7 Features Categóricas por Classe

In [ ]:
cat_cols = cfg["features"]["categorical_features"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for idx, col in enumerate(cat_cols):
    ax = axes.ravel()[idx]
    ctab = pd.crosstab(train_df[col], train_df["label"].map(cfg["physics"]["class_labels"]), normalize="columns")
    ctab.plot(kind="bar", ax=ax, alpha=0.85, width=0.8)
    ax.set_title(col.replace("reg_dnn_moe_", ""), fontsize=10, fontweight="bold")
    ax.set_ylabel("Fração na classe")
    ax.grid(alpha=0.3, axis="y")
    if idx != 0:
        ax.get_legend().remove()
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

## 6. Poder Discriminante Univariado (Teste de Sanidade 13.3) e Correlações

Confere se alguma variável isolada separa quase perfeitamente sinal de fundo ($|\text{AUC} - 0.5| > 0.40$), o que indicaria artefato de seleção ou vazamento.

In [ ]:
uni_auc_df = univariate_feature_aucs(
    train_df=train_df,
    x_cont_train=data.x_cont["train"],
    x_cat_train=data.x_cat["train"],
    continuous_features=cont_cols,
    cat_vocab_sizes=data.cat_preprocessor.vocab_sizes,
    seed=cfg["seed"],
)

print("Top 12 variáveis mais discriminantes individualmente (HH vs Fundo):")
display(uni_auc_df.head(12))

susp = uni_auc_df[uni_auc_df["abs_delta_0p5"] > 0.40]
print(
    "ATENÇÃO: features quase perfeitas:" if len(susp) else "Teste 13.3 OK: nenhuma feature isolada separa trivialmente sinal e fundo.",
    susp["feature"].tolist(),
)

# Matriz de correlação linear entre as 15 features contínuas mais discriminantes
top_cont = [f for f in uni_auc_df["feature"] if not f.startswith("cat:")][:15]
top_idx = [cont_cols.index(f) for f in top_cont]
sub_rng = np.random.default_rng(cfg["seed"]).choice(len(x_tr), size=min(100_000, len(x_tr)), replace=False)
corr = np.corrcoef(x_tr[sub_rng][:, top_idx], rowvar=False)

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
short_names = [f.replace("reg_dnn_moe_", "") for f in top_cont]
ax.set_xticks(np.arange(len(short_names)))
ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=8.5)
ax.set_yticks(np.arange(len(short_names)))
ax.set_yticklabels(short_names, fontsize=8.5)
ax.set_title("Matriz de Correlação — Top 15 Features Contínuas")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()